# Importing Necessary Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier

from scipy.sparse import hstack
from tqdm import tqdm

In [2]:
train_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [3]:
print("Dataset preview : \n")
train_data.head()

Dataset preview : 



,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
print(f"Train data size : {train_data.shape}")
print(f"Test data size : {test_data.shape}")

Train data size : (2000, 8)
Test data size : (500, 7)


# Preprocessing

In [5]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = text.replace("\n"," ")
    text = " ".join(text.split())
    return text

In [6]:
for col in ["prompt","A","B","C","D","E"]:
    train_data[col] = train_data[col].fillna("").apply(clean_text)
    test_data[col] = test_data[col].fillna("").apply(clean_text)

# Convert MCQ into Ranking Format

In [7]:
options = ["A","B","C","D","E"]

train_rows = []

for _, row in tqdm(train_data.iterrows(), total=len(train_data)):
    
    correct = row["answer"]
    
    for op in options:
        
        train_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "option": row[op],
            "label": int(op == correct)
        })

train_long = pd.DataFrame(train_rows)

print(train_long.shape)

100%|██████████| 2000/2000 [00:00<00:00, 12375.01it/s]

(10000, 4)


# Test Ranking Format

In [8]:
test_rows = []

for _, row in tqdm(test_data.iterrows(), total=len(test_data)):
    
    for op in options:
        
        test_rows.append({
            "id": row["id"],
            "option_label": op,
            "prompt": row["prompt"],
            "option": row[op]
        })

test_long = pd.DataFrame(test_rows)

100%|██████████| 500/500 [00:00<00:00, 13239.51it/s]


# TF-IDF Feature Engineering

In [9]:
train_text = (
    train_long["prompt"] +
    " [SEP] " +
    train_long["option"]
)

test_text = (
    test_long["prompt"] +
    " [SEP] " +
    test_long["option"]
)

In [10]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    stop_words="english"
)

In [11]:
X_train = tfidf.fit_transform(train_text)

X_test = tfidf.transform(test_text)

y = train_long["label"]

In [12]:
print(type(X_train))
print(type(X_test))

<class 'scipy.sparse._csr.csr_matrix'>
<class 'scipy.sparse._csr.csr_matrix'>


In [13]:
question_ids = train_long["id"]

# LightGBM

In [14]:
lgb_params = {
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "objective": "binary",
    "metric": "auc",
    "verbosity": -1,
    "random_state": 42
}

## Train

In [15]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(train_long))
test_preds = np.zeros(len(test_long))

for fold, (tr_idx, val_idx) in enumerate(
        skf.split(X_train, y)):
    
    print(f"Fold {fold+1}")
    
    model = LGBMClassifier(**lgb_params)
    
    model.fit(
        X_train[tr_idx],
        y.iloc[tr_idx]
    )
    
    oof[val_idx] = model.predict_proba(
        X_train[val_idx]
    )[:,1]
    
    test_preds += (
        model.predict_proba(X_test)[:,1] / 5
    )
    print(test_preds)

Fold 1


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[1.99865901e-01 3.74639383e-06 1.01218153e-06 ... 1.98407497e-01
 1.22145463e-04 2.24206886e-05]
Fold 2


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[3.99741452e-01 3.45476400e-05 7.21003728e-06 ... 3.96903209e-01
 2.68601986e-04 1.53064347e-04]
Fold 3


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[5.99178794e-01 5.56646255e-05 8.68338982e-06 ... 4.21170266e-01
 3.74500595e-04 1.72817525e-04]
Fold 4


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[7.99010657e-01 9.58656682e-05 1.69237696e-05 ... 6.20302066e-01
 6.78611478e-04 2.27342619e-04]
Fold 5


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[9.98438711e-01 1.04953391e-04 1.82582846e-05 ... 8.19262375e-01
 7.66911130e-04 2.86015993e-04]


In [16]:
test_long["score"] = test_preds

In [17]:
test_long.head()

,id,option_label,prompt,option,score
0,1,A,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",0.998439
1,1,B,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",0.000105
2,1,C,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",0.000018
3,1,D,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",0.000015
4,1,E,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",0.000156


# Submission

In [18]:
submission_rows = []

for qid, group in test_long.groupby("id"):
    
    ranked = group.sort_values(
        "score",
        ascending=False
    )
    
    top3 = ranked["option_label"].head(3).tolist()
    
    submission_rows.append({
        "id": qid,
        "prediction": " ".join(top3)
    })

In [19]:
submission = pd.DataFrame(submission_rows)
submission.head()

,id,prediction
0,1,A E B
1,2,B E D
2,3,B D E
3,4,E C D
4,5,C D B


In [20]:
submission.to_csv(
    "submission.csv",
    index=False
)